### 11. Multimodal Data Integration

This notebook integrates the demographic, questionnaire, and wearable participant-level datasets to create the multimodal datasets required for the subsequent modeling experiments.

#### 1. Libraries and Project Paths

In [78]:
# =============================================================================
# Libraries and project paths
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# =============================================================================
# Locate project root
# =============================================================================

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/ and src/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ===========================================================
# Import shared modeling utilities
# ===========================================================
from src.modeling import outputs

# =============================================================================
# Import shared feature definitions
# =============================================================================

from src.modeling.feature_sets import (
    DEMOGRAPHIC_FEATURES,
    QUESTIONNAIRE_FEATURES,
    METADATA_COLUMNS,
    get_wearable_features,
)

# =============================================================================
# Define data paths
# =============================================================================

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

DEMOGRAPHICS_FILE = (
    PROCESSED_DIR
    / "demographics_clean.csv"
)

QUESTIONNAIRE_FILE = (
    PROCESSED_DIR
    / "questionnaire_cleaned.csv"
)

WEARABLE_FILE = (
    PROCESSED_DIR
    / "wearable_features.csv"
)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Demographics file:", DEMOGRAPHICS_FILE)
print("Questionnaire file:", QUESTIONNAIRE_FILE)
print("Wearable file:", WEARABLE_FILE)

Project root: c:\Users\Main\Documents\GitHub\AI-Assisted-Screening-of-Parkinson-s-Disease
Processed data directory: c:\Users\Main\Documents\GitHub\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed
Demographics file: c:\Users\Main\Documents\GitHub\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\demographics_clean.csv
Questionnaire file: c:\Users\Main\Documents\GitHub\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\questionnaire_cleaned.csv
Wearable file: c:\Users\Main\Documents\GitHub\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\wearable_features.csv


#### 2. Load Source Datasets

The cleaned demographic, questionnaire, and participant-level wearable feature datasets were loaded as the input sources for multimodal integration. 

Participant identifiers were loaded as strings to preserve their original formatting and ensure consistent matching across datasets.

In [27]:
# =============================================================================
# Verify source files
# =============================================================================

source_files = {
    "Demographics": DEMOGRAPHICS_FILE,
    "Questionnaire": QUESTIONNAIRE_FILE,
    "Wearable": WEARABLE_FILE,
}

for dataset_name, file_path in source_files.items():

    if not file_path.exists():
        raise FileNotFoundError(
            f"{dataset_name} dataset not found: {file_path}"
        )

print("All source dataset found")

All source dataset found


In [28]:
# =============================================================================
# Load source datasets
# =============================================================================

demographics_df = pd.read_csv(
    DEMOGRAPHICS_FILE,
    dtype={"patient_id": str},
)

questionnaire_df = pd.read_csv(
    QUESTIONNAIRE_FILE,
    dtype={"patient_id": str},
)

wearable_df = pd.read_csv(
    WEARABLE_FILE,
    dtype={"patient_id": str},
)

print(
    "Demographics shape:",
    demographics_df.shape
)

print(
    "Questionnaire shape:",
    questionnaire_df.shape
)

print(
    "Wearable shape:",
    wearable_df.shape
)

Demographics shape: (469, 15)
Questionnaire shape: (469, 46)
Wearable shape: (469, 183)


#### 3. Define and Validate Modality Features

The modality-specific predictor sets were obtained from the shared feature definitions to preserve consistency with the Week 4 unimodal baseline experiments.

In [29]:
# =============================================================================
# Feature set
# =============================================================================

WEARABLE_FEATURES = get_wearable_features(
    wearable_df
)

print(
    "Demographic predictors:",
    len(DEMOGRAPHIC_FEATURES)
)

print(
    "Questionnaire predictors:",
    len(QUESTIONNAIRE_FEATURES)
)

print(
    "Wearable predictors:",
    len(WEARABLE_FEATURES)
)

Demographic predictors: 9
Questionnaire predictors: 41
Wearable predictors: 182


In [36]:
# =============================================================================
# Verify feature availability in source datasets
# =============================================================================

missing_demographic_features = [
    feature
    for feature in DEMOGRAPHIC_FEATURES
    if feature not in demographics_df.columns
]

missing_questionnaire_features = [
    feature
    for feature in QUESTIONNAIRE_FEATURES
    if feature not in questionnaire_df.columns
]

missing_wearable_features = [
    feature
    for feature in WEARABLE_FEATURES
    if feature not in wearable_df.columns
]

print(
    "Missing demographic features:",
    len(missing_demographic_features)
)

print(
    "Missing questionnaire features:",
    len(missing_questionnaire_features)
)

print(
    "Missing wearable features:",
    len(missing_wearable_features)
)

Missing demographic features: 0
Missing questionnaire features: 0
Missing wearable features: 0


In [37]:
assert not missing_demographic_features, (
    f"Missing demographic features: "
    f"{missing_demographic_features}"
)

assert not missing_questionnaire_features, (
    f"Missing questionnaire features: "
    f"{missing_questionnaire_features}"
)

assert not missing_wearable_features, (
    f"Missing wearable features: "
    f"{missing_wearable_features}"
)

print("All expected modality features found")

All expected modality features found


#### Observartion:
The demographic modality contains 9 predictors, the questionnaire modality contains 41 predictors, and the wearable modality contains 182 participant-level features.

#### 4. Verify Participant IDs and Duplicates

Before integrating the three data modalities, participant identifiers were validated to ensure that the demographic, questionnaire, and wearable datasets contained the same participant population.

In [38]:
# =============================================================================
# Verify patient_id column
# =============================================================================

source_datasets = {
    "Demographics": demographics_df,
    "Questionnaire": questionnaire_df,
    "Wearable": wearable_df,
}

for dataset_name, dataframe in source_datasets.items():

    if "patient_id" not in dataframe.columns:
        raise ValueError(
            f"{dataset_name} dataset does not contain patient_id."
        )

print("Patient_id column found in all source datasets")

Patient_id column found in all source datasets


In [39]:
# =============================================================================
# Verify participant counts and duplicates
# =============================================================================

participant_validation = []

for dataset_name, dataframe in source_datasets.items():

    total_rows = len(dataframe)

    unique_participants = (
        dataframe["patient_id"].nunique()
    )

    duplicated_participants = (
        dataframe["patient_id"]
        .duplicated()
        .sum()
    )

    participant_validation.append({
        "Dataset": dataset_name,
        "Rows": total_rows,
        "Unique Participants": unique_participants,
        "Duplicated Participants": duplicated_participants,
    })


participant_validation_df = pd.DataFrame(
    participant_validation
)

participant_validation_df

,Dataset,Rows,Unique Participants,Duplicated Participants
0,Demographics,469,469,0
1,Questionnaire,469,469,0
2,Wearable,469,469,0


In [40]:
# =============================================================================
# Validate expected participant structure
# =============================================================================

assert (
    participant_validation_df["Unique Participants"]
    == 469
).all(), (
    "One or more datasets do not contain 469 unique participants."
)

assert (
    participant_validation_df["Duplicated Participants"]
    == 0
).all(), (
    "Duplicated participant IDs were found."
)

print("Participant counts and duplicates verified")

Participant counts and duplicates verified


In [41]:
# =============================================================================
# Verify participant ID consistency across datasets
# =============================================================================

demographic_ids = set(
    demographics_df["patient_id"]
)

questionnaire_ids = set(
    questionnaire_df["patient_id"]
)

wearable_ids = set(
    wearable_df["patient_id"]
)

same_demographics_questionnaire = (
    demographic_ids
    == questionnaire_ids
)

same_demographics_wearable = (
    demographic_ids
    == wearable_ids
)

same_questionnaire_wearable = (
    questionnaire_ids
    == wearable_ids
)


participant_id_consistency = pd.DataFrame({
    "Dataset Comparison": [
        "Demographics vs Questionnaire",
        "Demographics vs Wearable",
        "Questionnaire vs Wearable",
    ],
    "Same Participant IDs": [
        same_demographics_questionnaire,
        same_demographics_wearable,
        same_questionnaire_wearable,
    ],
})

participant_id_consistency

,Dataset Comparison,Same Participant IDs
0,Demographics vs Questionnaire,True
1,Demographics vs Wearable,True
2,Questionnaire vs Wearable,True


#### Observation: 
All three source datasets contained 469 unique participants with no duplicated participant identifiers. The participant ID sets were identical across the demographic, questionnaire, and wearable datasets, confirming that the three modalities can be safely integrated at the participant level using patient_id.

#### 5. Create Demographics + Questionnaire Dataset

In [42]:
# =============================================================================
# Select demographic and questionnaire columns
# =============================================================================

demographics_selected = demographics_df[
    [
        "patient_id",
        "condition_group",
        "label",
        *DEMOGRAPHIC_FEATURES,
    ]
].copy()

questionnaire_selected = questionnaire_df[
    [
        "patient_id",
        *QUESTIONNAIRE_FEATURES,
    ]
].copy()

print(
    "Demographics selected shape:",
    demographics_selected.shape
)

print(
    "Questionnaire selected shape:",
    questionnaire_selected.shape
)

Demographics selected shape: (469, 12)
Questionnaire selected shape: (469, 42)


In [43]:
# =============================================================================
# Merge demographics and questionnaire datasets
# =============================================================================

demographics_questionnaire_df = (
    demographics_selected
    .merge(
        questionnaire_selected,
        on="patient_id",
        how="inner",
        validate="one_to_one", # check that patient_id appears only once in each dataset
    )
)

print(
    "Demographics + Questionnaire shape:",
    demographics_questionnaire_df.shape
)

demographics_questionnaire_df.head()

Demographics + Questionnaire shape: (469, 53)


,patient_id,condition_group,label,age,age_at_diagnosis,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,Q09,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30,total_symptom_count,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count
0,1,Healthy Control,0,56.0,NaN,173.0,78.0,Male,Right,Yes,Yes,Unknown,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,Other Movement Disorder,2,81.0,69.0,193.0,104.0,Male,Right,No,Unknown,No Effect,1,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,1,1,0,1,0,1,0,1,0,12,2,1,0,1,0,1,1,0,3,3
2,3,Healthy Control,0,45.0,NaN,170.0,78.0,Female,Right,No,Unknown,Unknown,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,Parkinson's Disease,1,67.0,63.0,161.0,90.0,Female,Right,No,Unknown,No Effect,0,1,0,1,0,0,0,1,1,1,0,0,0,0,0,1,0,0,0,0,1,1,1,0,0,1,1,1,0,0,12,1,2,1,1,0,1,1,0,2,3
4,5,Parkinson's Disease,1,75.0,65.0,172.0,86.0,Male,Left,No,Unknown,Unknown,1,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1,0,0,0,11,2,1,0,0,0,1,0,0,2,5


In [44]:
# =============================================================================
# Verify integrated dataset structure
# =============================================================================

expected_features = (
    len(DEMOGRAPHIC_FEATURES)
    + len(QUESTIONNAIRE_FEATURES)
)

expected_columns = (3 + expected_features)

print("Expected predictors:",
    expected_features
)

print("Actual predictors:",
    demographics_questionnaire_df.shape[1] - 3
)

print("Expected total columns:",
    expected_columns
)

print("Actual total columns:",
    demographics_questionnaire_df.shape[1]
)

Expected predictors: 50
Actual predictors: 50
Expected total columns: 53
Actual total columns: 53


In [46]:
assert len(demographics_questionnaire_df) == 469, (
    "Demographics + Questionnaire dataset "
    "does not contain 469 participants."
)

assert demographics_questionnaire_df[
    "patient_id"
].nunique() == 469, (
    "Unexpected duplicated or missing participant IDs."
)

assert (
    demographics_questionnaire_df.shape[1]
    == expected_columns
), (
    f"Expected {expected_columns} total columns, "
    f"found {demographics_questionnaire_df.shape[1]}."
)

print(
    "Demographics + Questionnaire dataset verified"
)

Demographics + Questionnaire dataset verified


In [47]:
# =============================================================================
# Verify no duplicated participants after integration
# =============================================================================

duplicated_participants = (
    demographics_questionnaire_df[
        "patient_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicated participants:",
    duplicated_participants
)

assert duplicated_participants == 0

print(
    "No duplicated patient_id found"
)

Duplicated participants: 0
No duplicated patient_id found


In [54]:
# =====================================================
# Brief integration summary
# =====================================================

demographics_questionnaire_summary = pd.DataFrame({
    "Metric": [
        "Patient_id",
        "Demographic features",
        "Questionnaire features",
        "Total predictors",
        "Total columns",
        "Duplicated participants",
    ],
    "Value": [
        len(demographics_questionnaire_df),
        len(DEMOGRAPHIC_FEATURES),
        len(QUESTIONNAIRE_FEATURES),
        expected_features,
        demographics_questionnaire_df.shape[1],
        duplicated_participants,
    ],
})

demographics_questionnaire_summary

,Metric,Value
0,Patient_id,469
1,Demographic features,9
2,Questionnaire features,41
3,Total predictors,50
4,Total columns,53
5,Duplicated participants,0


#### Observation:
The Demographics + Questionnaire dataset was successfully created for all 469 participants. 

The integration retained 9 demographic predictors and 41 questionnaire predictors, resulting in 50 predictive features and 53 total columns including patient_id, condition_group, and label. 

No duplicated participants were introduced during the merge.

#### 6. Build Wearable + Questionnaire Dataset

In [49]:
# =============================================================================
# Select wearable, questionnaire, and target columns
# =============================================================================

wearable_selected = wearable_df[
    [
        "patient_id",
        *WEARABLE_FEATURES,
    ]
].copy()

questionnaire_selected = questionnaire_df[
    [
        "patient_id",
        *QUESTIONNAIRE_FEATURES,
    ]
].copy()

participant_labels = demographics_df[
    [
        "patient_id",
        "condition_group",
        "label",
    ]
].copy()

print(
    "Wearable selected shape:",
    wearable_selected.shape
)

print(
    "Questionnaire selected shape:",
    questionnaire_selected.shape
)

print(
    "Participant labels shape:",
    participant_labels.shape
)

Wearable selected shape: (469, 183)
Questionnaire selected shape: (469, 42)
Participant labels shape: (469, 3)


In [50]:
# =============================================================================
# Merge wearable and questionnaire datasets
# =============================================================================

wearable_questionnaire_df = (
    wearable_selected
    .merge(
        questionnaire_selected,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

In [51]:
wearable_questionnaire_df = (
    participant_labels
    .merge(
        wearable_questionnaire_df,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

print(
    "Wearable + Questionnaire shape:",
    wearable_questionnaire_df.shape
)

wearable_questionnaire_df.head()

Wearable + Questionnaire shape: (469, 226)


,patient_id,condition_group,label,time_Accelerometer_X_Mean,time_Accelerometer_X_Median,time_Accelerometer_X_Std,time_Accelerometer_X_Min,time_Accelerometer_X_Max,time_Accelerometer_X_Range,time_Accelerometer_X_IQR,time_Accelerometer_X_RMS,time_Accelerometer_X_Energy,time_Accelerometer_Y_Mean,time_Accelerometer_Y_Median,time_Accelerometer_Y_Std,time_Accelerometer_Y_Min,time_Accelerometer_Y_Max,time_Accelerometer_Y_Range,time_Accelerometer_Y_IQR,time_Accelerometer_Y_RMS,time_Accelerometer_Y_Energy,time_Accelerometer_Z_Mean,time_Accelerometer_Z_Median,time_Accelerometer_Z_Std,time_Accelerometer_Z_Min,time_Accelerometer_Z_Max,time_Accelerometer_Z_Range,time_Accelerometer_Z_IQR,time_Accelerometer_Z_RMS,time_Accelerometer_Z_Energy,time_Gyroscope_X_Mean,time_Gyroscope_X_Median,time_Gyroscope_X_Std,time_Gyroscope_X_Min,time_Gyroscope_X_Max,time_Gyroscope_X_Range,time_Gyroscope_X_IQR,time_Gyroscope_X_RMS,time_Gyroscope_X_Energy,time_Gyroscope_Y_Mean,time_Gyroscope_Y_Median,time_Gyroscope_Y_Std,time_Gyroscope_Y_Min,time_Gyroscope_Y_Max,time_Gyroscope_Y_Range,time_Gyroscope_Y_IQR,time_Gyroscope_Y_RMS,time_Gyroscope_Y_Energy,time_Gyroscope_Z_Mean,time_Gyroscope_Z_Median,time_Gyroscope_Z_Std,time_Gyroscope_Z_Min,time_Gyroscope_Z_Max,time_Gyroscope_Z_Range,time_Gyroscope_Z_IQR,time_Gyroscope_Z_RMS,time_Gyroscope_Z_Energy,time_Acc_Magnitude_Mean,time_Acc_Magnitude_Median,time_Acc_Magnitude_Std,time_Acc_Magnitude_Min,time_Acc_Magnitude_Max,time_Acc_Magnitude_Range,time_Acc_Magnitude_IQR,time_Acc_Magnitude_RMS,time_Acc_Magnitude_Energy,time_Gyro_Magnitude_Mean,time_Gyro_Magnitude_Median,time_Gyro_Magnitude_Std,time_Gyro_Magnitude_Min,time_Gyro_Magnitude_Max,time_Gyro_Magnitude_Range,time_Gyro_Magnitude_IQR,time_Gyro_Magnitude_RMS,time_Gyro_Magnitude_Energy,freq_n_samples,freq_original_sampling_frequency_hz,freq_sampling_frequency_hz,freq_resampled_n_samples,freq_uniform_time_start_s,freq_uniform_time_end_s,freq_acc_x_dominant_frequency,freq_acc_x_spectral_centroid,freq_acc_x_spectral_entropy,freq_acc_x_spectral_power,freq_acc_y_dominant_frequency,freq_acc_y_spectral_centroid,freq_acc_y_spectral_entropy,freq_acc_y_spectral_power,freq_acc_z_dominant_frequency,freq_acc_z_spectral_centroid,freq_acc_z_spectral_entropy,freq_acc_z_spectral_power,freq_acc_magnitude_dominant_frequency,freq_acc_magnitude_spectral_centroid,freq_acc_magnitude_spectral_entropy,freq_acc_magnitude_spectral_power,freq_gyro_x_dominant_frequency,freq_gyro_x_spectral_centroid,freq_gyro_x_spectral_entropy,freq_gyro_x_spectral_power,freq_gyro_y_dominant_frequency,freq_gyro_y_spectral_centroid,freq_gyro_y_spectral_entropy,freq_gyro_y_spectral_power,freq_gyro_z_dominant_frequency,freq_gyro_z_spectral_centroid,freq_gyro_z_spectral_entropy,freq_gyro_z_spectral_power,freq_gyro_magnitude_dominant_frequency,freq_gyro_magnitude_spectral_centroid,freq_gyro_magnitude_spectral_entropy,freq_gyro_magnitude_spectral_power,symmetry_Accelerometer_X_Mean_difference,symmetry_Accelerometer_X_Median_difference,symmetry_Accelerometer_X_Std_difference,symmetry_Accelerometer_X_Min_difference,symmetry_Accelerometer_X_Max_difference,symmetry_Accelerometer_X_Range_difference,symmetry_Accelerometer_X_IQR_difference,symmetry_Accelerometer_X_RMS_difference,symmetry_Accelerometer_X_Energy_difference,symmetry_Accelerometer_Y_Mean_difference,symmetry_Accelerometer_Y_Median_difference,symmetry_Accelerometer_Y_Std_difference,symmetry_Accelerometer_Y_Min_difference,symmetry_Accelerometer_Y_Max_difference,symmetry_Accelerometer_Y_Range_difference,symmetry_Accelerometer_Y_IQR_difference,symmetry_Accelerometer_Y_RMS_difference,symmetry_Accelerometer_Y_Energy_difference,symmetry_Accelerometer_Z_Mean_difference,symmetry_Accelerometer_Z_Median_difference,symmetry_Accelerometer_Z_Std_difference,symmetry_Accelerometer_Z_Min_difference,symmetry_Accelerometer_Z_Max_difference,symmetry_Accelerometer_Z_Range_difference,symmetry_Accelerometer_Z_IQR_difference,symmetry_Accelerometer_Z_RMS_difference,symmetry_Accelerometer_Z_Ene

In [52]:
# =============================================================================
# Verify integrated dataset structure
# =============================================================================

expected_wq_features = (
    len(WEARABLE_FEATURES)
    + len(QUESTIONNAIRE_FEATURES)
)

expected_wq_columns = (3 + expected_wq_features)

print("Expected predictors:",
    expected_wq_features
)

print("Actual predictors:",
    wearable_questionnaire_df.shape[1] - 3
)

print("Expected total columns:",
    expected_wq_columns
)

print("Actual total columns:",
    wearable_questionnaire_df.shape[1]
)

Expected predictors: 223
Actual predictors: 223
Expected total columns: 226
Actual total columns: 226


In [53]:
# =============================================================================
# Verify participant structure
# =============================================================================

assert len(wearable_questionnaire_df) == 469, (
    "Wearable + Questionnaire dataset "
    "does not contain 469 participants."
)

assert wearable_questionnaire_df[
    "patient_id"
].nunique() == 469, (
    "Unexpected missing or duplicated participant IDs."
)

duplicated_wq_participants = (
    wearable_questionnaire_df[
        "patient_id"
    ]
    .duplicated()
    .sum()
)

assert duplicated_wq_participants == 0, (
    "Duplicated participants were introduced during integration."
)

assert (
    wearable_questionnaire_df.shape[1]
    == expected_wq_columns
), (
    f"Expected {expected_wq_columns} total columns, "
    f"found {wearable_questionnaire_df.shape[1]}."
)

print(
    "Wearable + Questionnaire dataset verified"
)

Wearable + Questionnaire dataset verified


In [55]:
wearable_questionnaire_summary = pd.DataFrame({
    "Metric": [
        "Patient_id",
        "Wearable features",
        "Questionnaire features",
        "Total predictors",
        "Total columns",
        "Duplicated participants",
    ],
    "Value": [
        len(wearable_questionnaire_df),
        len(WEARABLE_FEATURES),
        len(QUESTIONNAIRE_FEATURES),
        expected_wq_features,
        wearable_questionnaire_df.shape[1],
        duplicated_wq_participants,
    ],
})

wearable_questionnaire_summary

,Metric,Value
0,Patient_id,469
1,Wearable features,182
2,Questionnaire features,41
3,Total predictors,223
4,Total columns,226
5,Duplicated participants,0


#### Observation: 

The Wearable + Questionnaire dataset was successfully created for all 469 participants. The integration combined 182 wearable predictors and 41 questionnaire predictors, resulting in 223 predictive features

#### 7. Build Full Multimodal Dataset

The three participant-level modalities were integrated to create the full multimodal dataset used for subsequent modeling experiments.

In [56]:
# =============================================================================
# Select columns for full multimodal integration
# =============================================================================

demographics_full_selected = demographics_df[
    [
        "patient_id",
        "condition_group",
        "label",
        *DEMOGRAPHIC_FEATURES,
    ]
].copy()

questionnaire_full_selected = questionnaire_df[
    [
        "patient_id",
        *QUESTIONNAIRE_FEATURES,
    ]
].copy()

wearable_full_selected = wearable_df[
    [
        "patient_id",
        *WEARABLE_FEATURES,
    ]
].copy()

In [57]:
# =============================================================================
# Merge the three modalities
# =============================================================================

# Merge demographics and questionnaire
multimodal_full_df = (
    demographics_full_selected
    .merge(
        questionnaire_full_selected,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

# Add wearable features
multimodal_full_df = (
    multimodal_full_df
    .merge(
        wearable_full_selected,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

print(
    "Full multimodal dataset shape:",
    multimodal_full_df.shape
)

multimodal_full_df.head()

Full multimodal dataset shape: (469, 235)


,patient_id,condition_group,label,age,age_at_diagnosis,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,Q09,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30,total_symptom_count,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count,time_Accelerometer_X_Mean,time_Accelerometer_X_Median,time_Accelerometer_X_Std,time_Accelerometer_X_Min,time_Accelerometer_X_Max,time_Accelerometer_X_Range,time_Accelerometer_X_IQR,time_Accelerometer_X_RMS,time_Accelerometer_X_Energy,time_Accelerometer_Y_Mean,time_Accelerometer_Y_Median,time_Accelerometer_Y_Std,time_Accelerometer_Y_Min,time_Accelerometer_Y_Max,time_Accelerometer_Y_Range,time_Accelerometer_Y_IQR,time_Accelerometer_Y_RMS,time_Accelerometer_Y_Energy,time_Accelerometer_Z_Mean,time_Accelerometer_Z_Median,time_Accelerometer_Z_Std,time_Accelerometer_Z_Min,time_Accelerometer_Z_Max,time_Accelerometer_Z_Range,time_Accelerometer_Z_IQR,time_Accelerometer_Z_RMS,time_Accelerometer_Z_Energy,time_Gyroscope_X_Mean,time_Gyroscope_X_Median,time_Gyroscope_X_Std,time_Gyroscope_X_Min,time_Gyroscope_X_Max,time_Gyroscope_X_Range,time_Gyroscope_X_IQR,time_Gyroscope_X_RMS,time_Gyroscope_X_Energy,time_Gyroscope_Y_Mean,time_Gyroscope_Y_Median,time_Gyroscope_Y_Std,time_Gyroscope_Y_Min,time_Gyroscope_Y_Max,time_Gyroscope_Y_Range,time_Gyroscope_Y_IQR,time_Gyroscope_Y_RMS,time_Gyroscope_Y_Energy,time_Gyroscope_Z_Mean,time_Gyroscope_Z_Median,time_Gyroscope_Z_Std,time_Gyroscope_Z_Min,time_Gyroscope_Z_Max,time_Gyroscope_Z_Range,time_Gyroscope_Z_IQR,time_Gyroscope_Z_RMS,time_Gyroscope_Z_Energy,time_Acc_Magnitude_Mean,time_Acc_Magnitude_Median,time_Acc_Magnitude_Std,time_Acc_Magnitude_Min,time_Acc_Magnitude_Max,time_Acc_Magnitude_Range,time_Acc_Magnitude_IQR,time_Acc_Magnitude_RMS,time_Acc_Magnitude_Energy,time_Gyro_Magnitude_Mean,time_Gyro_Magnitude_Median,time_Gyro_Magnitude_Std,time_Gyro_Magnitude_Min,time_Gyro_Magnitude_Max,time_Gyro_Magnitude_Range,time_Gyro_Magnitude_IQR,time_Gyro_Magnitude_RMS,time_Gyro_Magnitude_Energy,freq_n_samples,freq_original_sampling_frequency_hz,freq_sampling_frequency_hz,freq_resampled_n_samples,freq_uniform_time_start_s,freq_uniform_time_end_s,freq_acc_x_dominant_frequency,freq_acc_x_spectral_centroid,freq_acc_x_spectral_entropy,freq_acc_x_spectral_power,freq_acc_y_dominant_frequency,freq_acc_y_spectral_centroid,freq_acc_y_spectral_entropy,freq_acc_y_spectral_power,freq_acc_z_dominant_frequency,freq_acc_z_spectral_centroid,freq_acc_z_spectral_entropy,freq_acc_z_spectral_power,freq_acc_magnitude_dominant_frequency,freq_acc_magnitude_spectral_centroid,freq_acc_magnitude_spectral_entropy,freq_acc_magnitude_spectral_power,freq_gyro_x_dominant_frequency,freq_gyro_x_spectral_centroid,freq_gyro_x_spectral_entropy,freq_gyro_x_spectral_power,freq_gyro_y_dominant_frequency,freq_gyro_y_spectral_centroid,freq_gyro_y_spectral_entropy,freq_gyro_y_spectral_power,freq_gyro_z_dominant_frequency,freq_gyro_z_spectral_centroid,freq_gyro_z_spectral_entropy,freq_gyro_z_spectral_power,freq_gyro_magnitude_dominant_frequency,freq_gyro_magnitude_spectral_centroid,freq_gyro_magnitude_spectral_entropy,freq_gyro_magnitude_spectral_power,symmetry_Accelerometer_X_Mean_difference,symmetry_Accelerometer_X_Median_difference,symmetry_Accelerometer_X_Std_difference,symmetry_Accelerometer_X_Min_difference,symmetry_Accelerometer_X_Max_difference,symmetry_Accelerometer_X_Range_difference,symmetry_Accelerometer_X_IQR_difference,symmetry_Accelerometer_X_RMS_difference,symmetry_Accelerometer_X_Energy_difference,symmetry_Accelerometer_Y_Mean_difference,symmetry_Accelerometer_Y_Median_difference,symmetry_Accelerometer_Y_Std_difference,symmetry_Accelerometer_Y_Min_difference,symmetry_Accelerometer_Y_Max_difference,symmetry_Accelerometer_Y_Range_diff

In [59]:
# =============================================================================
# Verify full multimodal feature structure
# =============================================================================

expected_full_features = (
    len(DEMOGRAPHIC_FEATURES)
    + len(QUESTIONNAIRE_FEATURES)
    + len(WEARABLE_FEATURES)
)

expected_full_columns = (3 + expected_full_features)

actual_full_features = (multimodal_full_df.shape[1] - 3)

print("Expected predictors:",
    expected_full_features
)

print("Actual predictors:",
    actual_full_features
)

print("Expected total columns:",
    expected_full_columns
)

print("Actual total columns:",
    multimodal_full_df.shape[1]
)

Expected predictors: 232
Actual predictors: 232
Expected total columns: 235
Actual total columns: 235


In [61]:
# =============================================================================
# Verify participant structure
# =============================================================================

assert len(multimodal_full_df) == 469, (
    "Full multimodal dataset does not contain "
    "469 participants."
)

assert multimodal_full_df[
    "patient_id"
].nunique() == 469, (
    "Unexpected missing or duplicated participant IDs."
)

duplicated_full_participants = (
    multimodal_full_df[
        "patient_id"
    ]
    .duplicated()
    .sum()
)

assert duplicated_full_participants == 0, (
    "Duplicated participants were introduced "
    "during multimodal integration."
)

assert multimodal_full_df.shape[1] == expected_full_columns, (
    f"Expected {expected_full_columns} total columns, "
    f"found {multimodal_full_df.shape[1]}."
)

print(
    "Full multimodal dataset verified"
)

Full multimodal dataset verified


In [63]:
# ======================================================================
# Integration summary
# ======================================================================

full_multimodal_summary = pd.DataFrame({
    "Metric": [
        "Patient_id",
        "Demographic features",
        "Questionnaire features",
        "Wearable features",
        "Total predictors",
        "Total columns",
        "Duplicated participants",
    ],
    "Value": [
        len(multimodal_full_df),
        len(DEMOGRAPHIC_FEATURES),
        len(QUESTIONNAIRE_FEATURES),
        len(WEARABLE_FEATURES),
        expected_full_features,
        multimodal_full_df.shape[1],
        duplicated_full_participants,
    ],
})

full_multimodal_summary

,Metric,Value
0,Patient_id,469
1,Demographic features,9
2,Questionnaire features,41
3,Wearable features,182
4,Total predictors,232
5,Total columns,235
6,Duplicated participants,0


#### 8. Verify Class Distribution

In [64]:
# =============================================================================
# Define integrated datasets
# =============================================================================

integrated_datasets = {
    "Demographics + Questionnaire": demographics_questionnaire_df,
    "Wearable + Questionnaire": wearable_questionnaire_df,
    "Full Multimodal": multimodal_full_df,
}

In [65]:
# =============================================================================
# Calculate class distribution
# =============================================================================

class_distribution_rows = []

for dataset_name, dataframe in integrated_datasets.items():

    class_counts = (
        dataframe["condition_group"]
        .value_counts()
    )

    row = {
        "Dataset": dataset_name,
        "Patient_id": len(dataframe),
    }

    for class_name, count in class_counts.items():
        row[class_name] = count

    class_distribution_rows.append(row)


class_distribution_df = pd.DataFrame(
    class_distribution_rows
)

class_distribution_df

,Dataset,Patient_id,Parkinson's Disease,Other Movement Disorder,Healthy Control
0,Demographics + Questionnaire,469,276,114,79
1,Wearable + Questionnaire,469,276,114,79
2,Full Multimodal,469,276,114,79


#### Observation: 

The three integrated datasets retained the same 469 participants and identical diagnostic class distributions.

#### 9. Verify No Data Leakage

In [67]:
# =============================================================================
# Define non-predictor columns
# =============================================================================

NON_PREDICTOR_COLUMNS = [
    "patient_id",
    "condition_group",
    "label",
]

# =============================================================================
# Define expected predictor sets
# =============================================================================

expected_predictor_sets = {
    "Demographics + Questionnaire": (
        DEMOGRAPHIC_FEATURES
        + QUESTIONNAIRE_FEATURES
    ),

    "Wearable + Questionnaire": (
        WEARABLE_FEATURES
        + QUESTIONNAIRE_FEATURES
    ),

    "Full Multimodal": (
        DEMOGRAPHIC_FEATURES
        + QUESTIONNAIRE_FEATURES
        + WEARABLE_FEATURES
    ),
}

In [68]:
# =============================================================================
# Check for leakage columns in predictor sets
# =============================================================================

leakage_rows = []

for dataset_name, predictor_columns in expected_predictor_sets.items():

    leakage_columns = [
        column
        for column in NON_PREDICTOR_COLUMNS
        if column in predictor_columns
    ]

    leakage_rows.append({
        "Dataset": dataset_name,
        "Leakage Columns": (
            ", ".join(leakage_columns)
            if leakage_columns
            else "None"
        ),
        "Leakage Detected": (
            len(leakage_columns) > 0
        ),
    })


data_leakage_check = pd.DataFrame(
    leakage_rows
)

data_leakage_check

,Dataset,Leakage Columns,Leakage Detected
0,Demographics + Questionnaire,None,False
1,Wearable + Questionnaire,None,False
2,Full Multimodal,None,False


In [70]:
# =============================================================================
# Validate absence of leakage columns
# =============================================================================

assert not data_leakage_check[
    "Leakage Detected"
].any(), (
    "Potential target or identifier leakage was detected."
)

print(
    "No identifier or target Leakage detected "
    "in the predictor sets"
)

No identifier or target Leakage detected in the predictor sets


In [71]:
# =============================================================================
# Verify predictor columns against actual integrated datasets
# =============================================================================

integrated_predictor_validation = []

for dataset_name, dataframe in integrated_datasets.items():

    expected_predictors = set(
        expected_predictor_sets[dataset_name]
    )

    actual_predictors = set(
        dataframe.columns
    ) - set(NON_PREDICTOR_COLUMNS)

    same_predictors = (
        actual_predictors
        == expected_predictors
    )

    integrated_predictor_validation.append({
        "Dataset": dataset_name,
        "Expected Predictors": len(expected_predictors),
        "Actual Predictors": len(actual_predictors),
        "Predictor Set Match": same_predictors,
    })


predictor_validation_df = pd.DataFrame(
    integrated_predictor_validation
)

predictor_validation_df

,Dataset,Expected Predictors,Actual Predictors,Predictor Set Match
0,Demographics + Questionnaire,50,50,True
1,Wearable + Questionnaire,223,223,True
2,Full Multimodal,232,232,True


In [73]:
assert predictor_validation_df[
    "Predictor Set Match"
].all(), (
    "One or more integrated datasets contain "
    "unexpected predictor columns."
)

print(
    "Predictor sets verified "
    "with no unexpected Leakage columns"
)

Predictor sets verified with no unexpected Leakage columns


#### 10. Create Dataset Summary Table

This summary provides a final overview of the datasets prepared for the subsequent multimodal modeling experiments.

The table reports the number of patient_id, predictor features, and diagnostic class distribution for each integrated dataset.

In [74]:
# =============================================================================
# Create multimodal dataset summary
# =============================================================================

summary_rows = []

for dataset_name, dataframe in integrated_datasets.items():

    class_counts = (
        dataframe["condition_group"]
        .value_counts()
    )

    row = {
        "Dataset": dataset_name,
        "Patient_id": dataframe["patient_id"].nunique(),
        "Features": len(
            expected_predictor_sets[dataset_name]
        ),
    }

    # Add class distribution
    for class_name, count in class_counts.items():
        row[class_name] = count

    summary_rows.append(row)


dataset_summary = pd.DataFrame(
    summary_rows
)

dataset_summary

,Dataset,Patient_id,Features,Parkinson's Disease,Other Movement Disorder,Healthy Control
0,Demographics + Questionnaire,469,50,276,114,79
1,Wearable + Questionnaire,469,223,276,114,79
2,Full Multimodal,469,232,276,114,79


#### 11. Save and Verify Outputs

In [75]:
# =============================================================================
# Define multimodal dataset output paths
# =============================================================================

DEMOGRAPHICS_QUESTIONNAIRE_OUTPUT = (
    PROCESSED_DIR
    / "demographics_questionnaire.csv"
)

WEARABLE_QUESTIONNAIRE_OUTPUT = (
    PROCESSED_DIR
    / "wearable_questionnaire.csv"
)

MULTIMODAL_FULL_OUTPUT = (
    PROCESSED_DIR
    / "multimodal_full.csv"
)

In [76]:
# =============================================================================
# Save multimodal datasets
# =============================================================================

demographics_questionnaire_df.to_csv(
    DEMOGRAPHICS_QUESTIONNAIRE_OUTPUT,
    index=False,
)

wearable_questionnaire_df.to_csv(
    WEARABLE_QUESTIONNAIRE_OUTPUT,
    index=False,
)

multimodal_full_df.to_csv(
    MULTIMODAL_FULL_OUTPUT,
    index=False,
)

print("Multimodal datasets saved")

Multimodal datasets saved


In [79]:
# =============================================================================
# Save dataset summary table
# =============================================================================

outputs.save_table(
    dataset_summary,
    "multimodal_dataset_summary.csv",
)

print("Dataset summary table saved")

Dataset summary table saved


In [80]:
# =============================================================================
# Verify saved multimodal datasets
# =============================================================================

saved_dataset_files = {
    "Demographics + Questionnaire":
        DEMOGRAPHICS_QUESTIONNAIRE_OUTPUT,

    "Wearable + Questionnaire":
        WEARABLE_QUESTIONNAIRE_OUTPUT,

    "Full Multimodal":
        MULTIMODAL_FULL_OUTPUT,
}

for dataset_name, file_path in saved_dataset_files.items():

    assert file_path.exists(), (
        f"{dataset_name} output file was not saved."
    )

    assert file_path.stat().st_size > 0, (
        f"{dataset_name} output file is empty."
    )

print("All mutimodal dataset filed verified")

All mutimodal dataset filed verified


In [82]:
# =============================================================================
# Reload saved datasets
# =============================================================================

saved_demographics_questionnaire = pd.read_csv(
    DEMOGRAPHICS_QUESTIONNAIRE_OUTPUT,
    dtype={"patient_id": str},
)

saved_wearable_questionnaire = pd.read_csv(
    WEARABLE_QUESTIONNAIRE_OUTPUT,
    dtype={"patient_id": str},
)

saved_multimodal_full = pd.read_csv(
    MULTIMODAL_FULL_OUTPUT,
    dtype={"patient_id": str},
)

# =============================================================================
# Verify saved dataset dimensions
# =============================================================================

saved_dataset_validation = pd.DataFrame({
    "Dataset": [
        "Demographics + Questionnaire",
        "Wearable + Questionnaire",
        "Full Multimodal",
    ],
    "Participants": [
        saved_demographics_questionnaire[
            "patient_id"
        ].nunique(),

        saved_wearable_questionnaire[
            "patient_id"
        ].nunique(),

        saved_multimodal_full[
            "patient_id"
        ].nunique(),
    ],
    "Features": [
        saved_demographics_questionnaire.shape[1] - 3,
        saved_wearable_questionnaire.shape[1] - 3,
        saved_multimodal_full.shape[1] - 3,
    ],
    "Total Columns": [
        saved_demographics_questionnaire.shape[1],
        saved_wearable_questionnaire.shape[1],
        saved_multimodal_full.shape[1],
    ],
})

saved_dataset_validation

,Dataset,Participants,Features,Total Columns
0,Demographics + Questionnaire,469,50,53
1,Wearable + Questionnaire,469,223,226
2,Full Multimodal,469,232,235


In [83]:
# =============================================================================
# Final output verification
# =============================================================================

assert (
    saved_dataset_validation["Participants"]
    == 469
).all(), (
    "Unexpected participant count after saving."
)

assert (
    saved_dataset_validation["Features"].tolist()
    == [50, 223, 232]
), (
    "Unexpected feature count after saving."
)

print("Final output verification passed")

Final output verification passed


#### Conclusion:

The multimodal data integration process was successfully completed at the participant level. Three datasets were created by combining the demographic, questionnaire, and wearable feature sets: 
- demographics_questionnaire.csv (50 predictors)
- wearable_questionnaire.csv (223 predictors)
- multimodal_full.csv (232 predictors)

All three integrated datasets retained the same 469 participants and preserved the diagnostic class distribution of the source data. Participant IDs were verified during integration, and no duplicated participants were introduced.